# GATv2 — Final Retrain with the Frozen Recipe (HI-Small)

- **What:** the frozen GATv2 recipe (`Outputs/TUNING/GAT/recipe.json`, tuned on config 2) retrained on the **train + val graph** for the frozen number of epochs on the four feature configs with edge features in message passing — 2 (base / base), 4 (base+GFP / base+GFP), 5 (base / base+GFP), 6 (base / GFP only) — four seeds each; test scored once per run with the frozen threshold; test metrics reported as mean ± std over the four seeds of every config.
- **Protocol (`EXPERIMENTS.md` §2b / §3b):** cosine T_max unchanged (100), training stops at the frozen best epoch; one frozen threshold for every run (the mean of the two tuning seeds' validation thresholds); the epoch and the threshold were tuned on config 2 and are transferred to the other configs unchanged; every split is scored (val is training data here and is marked as such); a threshold-sensitivity table is reported, never used to choose.
- **Known limitation:** the train + val graph has 496 batches per epoch vs 372 in tuning, so the frozen epoch means ~33 % more update steps.
- **Everything shared** lives in `gnn_core.py` (`load_recipe`, `make_trainval_graph`, `run_final`, `seed_summary`). This notebook only sets the operator, the configs, the seeds and paths.


## 0. Environment (Kaggle)

- Pinned PyTorch / PyG stack known to work on Kaggle GPU — do not change versions or order.
- Then clone (or pull) the public repo so the committed `gnn_core.py` is the one imported.

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/sandrokhizanishvili/gnn_for_fraudulent_patterns.git'
REPO_DIR = '/kaggle/working/gnn_for_fraudulent_patterns'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline
sys.path.append(REPO_DIR)

## 1. Configuration

- The configs and seeds of the final retrain and the paths; the recipe itself comes from `recipe.json`, nothing is set by hand here.
- `OUT_ROOT` sits inside the repo clone, so finished runs are skipped on a re-run (a config-2 session can be followed by the rest).


In [ ]:
OPERATOR = 'gat'
CFGS     = [dict(mp='base', readout='base'),     # config 2 — the configuration the recipe was tuned on
            dict(mp='full', readout='full'),     # config 4 — base+GFP / base+GFP
            dict(mp='base', readout='full'),     # config 5 — base / base+GFP
            dict(mp='base', readout='gfp')]      # config 6 — base / GFP only
SEEDS    = (42, 43, 44, 45)                      # one frozen threshold for every run (mean of the tuning seeds' thresholds)

MP_DIRECTION      = 'in'
TEMPORAL_SAMPLING = False

DATA_DIR    = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn'
RECIPE_PATH = f'{REPO_DIR}/Outputs/TUNING/{OPERATOR.upper()}/recipe.json'
OUT_ROOT    = f'{REPO_DIR}/Outputs/FINAL/{OPERATOR.upper()}/retrain'      # mirrors Outputs/FINAL/GAT/retrain/ in the repo

import gnn_core as core
os.makedirs(OUT_ROOT, exist_ok=True)
print(f'torch {core.torch.__version__} | device {core.device} | gnn_core from {core.__file__}')


## 2. The frozen recipe

- `load_recipe` stops with a clear message if `recipe.json` is missing or incomplete (run `GAT_tuning.ipynb` to the end first).

In [ ]:
frozen = core.load_recipe(RECIPE_PATH)
assert frozen['operator'] == OPERATOR and frozen['cfg'] == CFGS[0], 'recipe.json belongs to another operator / config'
assert set(frozen['thresholds']) == {str(s) for s in SEEDS}, f'recipe.json holds thresholds for {list(frozen["thresholds"])}, not {SEEDS}'
RECIPE = frozen['recipe']


## 3. Load graphs and build the train + val training graph

- The usual three snapshots for scoring; the training graph is the val snapshot with every edge labelled and evaluated (asserted against the train snapshot), built in memory by `run_final` for every seed.
- Shown once here so the counts are visible: 4,061,789 training seeds, 3,379 laundering.

In [ ]:
graphs, col_sets, node_dim = core.load_graphs(DATA_DIR)
fit_preview = core.make_trainval_graph(graphs)
del fit_preview

## 4. Model

- The frozen recipe built once per config for inspection; `invariant_params` is the GATv2 family's constant for this recipe and must be identical across the configs (only the edge projections and the readout input widen).


In [ ]:
for cfg in CFGS:
    model = core.build_model(OPERATOR, node_dim, len(col_sets[cfg['mp']]), len(col_sets[cfg['readout']]),
                             MP_DIRECTION == 'bidirectional', None, RECIPE)
    if cfg is CFGS[0]:
        print(model)
    print(f"mp {cfg['mp']:>4} / readout {cfg['readout']:>4}: params {core.count_params(model):,} | "
          f"invariant_params {core.invariant_params(model):,}")
    del model
print(f'recipe {RECIPE}')


## 5. Retrain the runs

- One call per (config, seed): train + val graph → `best_epoch` epochs on the frozen cosine schedule → every split scored at the frozen threshold → `results.json`, `history.csv`, `curves.png`, `sensitivity.csv`, `predictions.csv`, `final.pt` under `OUT_ROOT/<run>/seed-<s>/`.
- A run whose `results.json` exists is skipped.


In [ ]:
rows = []
for cfg in CFGS:
    for seed in SEEDS:
        print('=' * 90)
        rows.append(core.run_final(OPERATOR, cfg, graphs, col_sets, node_dim, OUT_ROOT, frozen, seed,
                                   MP_DIRECTION, TEMPORAL_SAMPLING))
print('=' * 90)
print('retrain complete')


## 6. Summary over the seeds

- `batch_summary_retrain.csv`: one row per (config, seed) with every `<split>_<metric>` column (val = seen in training); `invariant_params` identical in every row (asserted).
- `seed_summary_retrain.csv`: one row per config, mean ± std over its four seeds — test F1 and PR-AUC are the headline numbers.
- Per config: test at the frozen threshold next to test at the threshold re-swept on the seen val rows (secondary, PR-AUC is threshold-free), and the sensitivity points (test F1 at the frozen threshold ± 0.05 / ± 0.10 and at 0.5) per seed.


In [ ]:
summary = core.summarize_batch(rows, OUT_ROOT, 'batch_summary_retrain.csv')
display(summary[['run', 'seed', 'stop_epoch', 'frozen_threshold'] + core.DISPLAY_COLS[2:]].round(4))

per_cfg = []
for run, cfg_rows in summary.groupby('run', sort=False):
    per_cfg.append(core.seed_summary(cfg_rows.to_dict('records'), OUT_ROOT, f'seed_summary_retrain_{run}.csv'))
seeds = core.pd.concat(per_cfg, ignore_index=True)
seeds.to_csv(f'{OUT_ROOT}/seed_summary_retrain.csv', index=False)
display(seeds[['run', 'n_seeds', 'test_f1_mean', 'test_f1_std', 'test_pr_auc_mean', 'test_pr_auc_std',
               'test_recall_at_5pct_mean', 'val_f1_mean', 'train_f1_mean']].round(4))

print('--- test at the frozen threshold (headline) vs at the threshold re-swept on the seen val rows (secondary); PR-AUC is threshold-free')
display(summary[['run', 'seed', 'frozen_threshold', 'test_f1', 'test_precision', 'test_recall', 'val_seen_threshold',
                 'seen_test_f1', 'seen_test_precision', 'seen_test_recall', 'test_pr_auc']].round(4))
display(seeds[['run', 'test_f1_mean', 'test_f1_std', 'seen_test_f1_mean', 'seen_test_f1_std',
               'test_pr_auc_mean', 'test_pr_auc_std']].round(4))

for run in summary['run'].unique():
    tables = []
    for seed in SEEDS:
        t = core.pd.read_csv(f'{OUT_ROOT}/{run}/seed-{seed}/sensitivity.csv')
        tables.append(t.set_index('point')['f1'].rename(f'seed {seed}'))
    print(f'--- sensitivity (test F1 by threshold point), {run}')
    display(core.pd.concat(tables, axis=1).round(4))


## 7. Download

- Zip `OUT_ROOT`; extract into `Outputs/FINAL/GAT/retrain/` in the repo and commit the small files (`final.pt` and `predictions.csv` stay out of git).


In [ ]:
import shutil
shutil.make_archive('/kaggle/working/Outputs_FINAL_GAT', 'zip', OUT_ROOT)